<div style="background: linear-gradient(135deg, #FF441F 0%, #c2340f 100%); padding: 48px 40px; border-radius: 12px; color: white; margin-bottom: 8px;">
  <h1 style="margin:0; font-size:2.2rem;">🛵 Rappi Competitive Intelligence</h1>
  <p style="margin:12px 0 0; font-size:1.1rem; opacity:.85;">Análisis competitivo automatizado: Rappi vs Uber Eats vs DiDi Food — México</p>
  <p style="margin:8px 0 0; font-size:.85rem; opacity:.65;">Caso técnico · AI Engineer · 2026</p>
</div>

---
## Índice
1. [Approach y Scope](#1)
2. [Arquitectura del Sistema](#2)
3. [Demo — Ejecución del Scraper](#3)
4. [Datos Recolectados — Overview](#4)
5. [Top 5 Insights Accionables](#5)
6. [Decisiones Técnicas y Desafíos](#6)
7. [Limitaciones y Next Steps](#7)

In [ ]:
# Setup — instalar dependencias si es necesario
import sys, os
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'

# Colores de marca
COLORS = {'rappi': '#FF441F', 'ubereats': '#06C167', 'didifood': '#FF6900'}
LABELS = {'rappi': 'Rappi', 'ubereats': 'Uber Eats', 'didifood': 'DiDi Food'}

print('✅ Setup completo')

<a id='1'></a>

---
# 1. Approach y Scope

## ¿Qué decidí scrapear y por qué?

### El problema de negocio
Rappi compite en precio, velocidad y experiencia contra Uber Eats y DiDi Food. Sin datos sistemáticos de la competencia, los equipos de Pricing y Strategy toman decisiones a ciegas o basadas en observaciones puntuales.

### Decisión de scope
En lugar de scrapear todo el catálogo (miles de restaurantes), definí un **conjunto de referencia controlado** que permite comparación directa y sin ambigüedad:

| Decisión | Elección | Justificación |
|---|---|---|
| **Producto de referencia** | McDonald's (Big Mac, Combo, McNuggets) + OXXO (Coca-Cola, Agua) | Disponible en las 3 plataformas con menú idéntico — elimina confounders de calidad o porción |
| **Cobertura geográfica** | 30 direcciones en 5 ciudades | Cubre el espectro completo: zonas premium, medias, populares y suburbanas |
| **Métricas clave** | Precio, delivery fee, service fee, ETA, descuentos | Son los factores que determinan la decisión de compra del usuario |
| **Plataformas** | Rappi, Uber Eats, DiDi Food | Los 3 actores principales del mercado mexicano de delivery |

### Cobertura geográfica

```
CDMX (15 zonas)         Guadalajara (6)      Monterrey (6)
├── Polanco              ├── Centro           ├── Centro
├── Condesa              ├── Zapopan          ├── San Pedro G.G.
├── Roma Norte           ├── Tlaquepaque      ├── Sur
├── Tepito               ├── Providencia      ├── Santa Catarina
├── Iztapalapa           ├── Chapalita        ├── Apodaca
├── Santa Fe             └── Tonalá           └── Escobedo
├── Ecatepec (x3)
└── ...                  Puebla (1)           Cancún (2)
```

**¿Por qué zonas periféricas?** Ecatepec, Apodaca y Escobedo son zonas de mayor crecimiento para delivery — donde la competencia de DiDi es más agresiva y la sensibilidad al precio es máxima.

In [ ]:
# Visualizar la distribución de zonas en el dataset
with open('config/addresses.json', encoding='utf-8') as f:
    addresses = json.load(f)
with open('config/products.json', encoding='utf-8') as f:
    products = json.load(f)

df_addr = pd.DataFrame(addresses)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Direcciones por ciudad', 'Distribución por tipo de zona'],
    specs=[[{'type': 'xy'}, {'type': 'domain'}]])

city_counts = df_addr['city'].value_counts()
fig.add_trace(go.Bar(
    x=city_counts.index, y=city_counts.values,
    marker_color='#FF441F', text=city_counts.values,
    textposition='outside', showlegend=False), row=1, col=1)

zone_counts = df_addr['zone_type'].value_counts()
zone_labels_map = {
    'high_income': 'Alto ingreso', 'mid_high': 'Medio-alto',
    'mid': 'Medio', 'popular': 'Popular',
    'suburban': 'Suburbano', 'popular_suburban': 'Sub. popular'
}
fig.add_trace(go.Pie(
    labels=[zone_labels_map.get(z, z) for z in zone_counts.index],
    values=zone_counts.values,
    hole=0.4, showlegend=True), row=1, col=2)

fig.update_layout(height=380, title_text=f'Cobertura geográfica — {len(addresses)} direcciones en {df_addr["city"].nunique()} ciudades')
fig.show()

print(f'\n📍 {len(addresses)} direcciones · {df_addr["city"].nunique()} ciudades · {len(products)} productos de referencia')
print(f'📊 Total de observaciones posibles: {len(addresses)} × {len(products)} × 3 plataformas = {len(addresses)*len(products)*3} registros')

<a id='2'></a>

---
# 2. Arquitectura del Sistema

## Diseño en capas

```
┌─────────────────────────────────────────────────────┐
│                    main.py (CLI)                    │  argparse · mock/live toggle · output
└────────────────────────┬────────────────────────────┘
                         │
         ┌───────────────┼───────────────┐
         ▼               ▼               ▼
   RappiScraper   UberEatsScraper   DiFoodScraper
         └───────────────┼───────────────┘
                         │ hereda de
                    BaseScraper
              (retry · rate limit · stealth)
                         │
                         ▼
               mock_data.py (fallback)
                         │
                         ▼
              data/ (JSON + CSV + logs)
                         │
                         ▼
           generate_report.py → informe.html
```

## Clase base — `BaseScraper`

Todos los scrapers heredan de una clase abstracta que provee:
- **Retry logic** con backoff exponencial (hasta 3 intentos)
- **Rate limiting** mínimo de 2s entre requests por dominio
- **Stealth anti-detección** via Playwright (user-agent real, sin `navigator.webdriver`, locale mexicano)
- **Screenshots** automáticos como evidencia
- **Context manager** `async with scraper as s` para lifecycle limpio del browser

In [ ]:
# Mostrar la interfaz pública de BaseScraper
import inspect
from scrapers.base import BaseScraper, ScrapingResult

public_methods = [
    (name, inspect.getdoc(method))
    for name, method in inspect.getmembers(BaseScraper, predicate=inspect.isfunction)
    if not name.startswith('__')
]

print('BaseScraper — métodos disponibles:\n')
print(f'{"Método":<30} {"Descripción"}')
print('─' * 75)
for name, doc in public_methods:
    desc = doc.split('\n')[0] if doc else '—'
    print(f'{name:<30} {desc}')

In [ ]:
# Mostrar el modelo de datos — ScrapingResult
r = ScrapingResult(
    platform='rappi',
    address={'id': 'cdmx_polanco', 'city': 'CDMX', 'zone': 'polanco',
             'zone_type': 'high_income', 'full_address': 'Av. Masaryk 61, Polanco'},
    product={'id': 'bigmac', 'name': 'Big Mac', 'category': 'fast_food',
             'restaurant': "McDonald's", 'reference_price_mxn': 95}
)
r.price_product = 107.50
r.delivery_fee = 29.0
r.service_fee = 8.60
r.estimated_delivery_min = 22
r.estimated_delivery_max = 32
r.discounts_active = ['Envío gratis con Rappi Prime', '30% en McDonald\'s']
r.final_price_total = 145.10

print('Ejemplo de ScrapingResult serializado:\n')
for k, v in r.to_dict().items():
    print(f'  {k:<30} {v}')

<a id='3'></a>

---
# 3. Demo — Ejecución del Scraper

## Modos de ejecución

El sistema tiene dos modos:

| Modo | Comando | Cuándo usarlo |
|---|---|---|
| **Mock** (default) | `python main.py` | Demo, testing, desarrollo — datos realistas sin riesgo de bloqueos |
| **Live** | `python main.py --live` | Scraping real con Playwright — requiere IP mexicana para cobertura completa |

**¿Por qué un modo mock?** Las plataformas (especialmente Uber Eats con Cloudflare) bloquean IPs no mexicanas y detectan headless browsers. El mock data fue diseñado con valores basados en investigación pública para ser realista y reproducible.

### Ejecutamos el pipeline completo:

In [ ]:
# Ejecutar el scraper en modo mock (3 plataformas × 30 dirs × 5 productos)
from scrapers.mock_data import generate_platform_data
from pathlib import Path
from datetime import datetime

platforms = ['rappi', 'ubereats', 'didifood']
all_results = []

for platform in platforms:
    print(f'⚙️  Scraping {platform}...', end=' ')
    results = generate_platform_data(platform, addresses, products)
    all_results.extend(results)
    available = sum(1 for r in results if r['restaurant_available'])
    print(f'{len(results)} registros generados ({available} con datos válidos)')

df = pd.DataFrame(all_results)
df_valid = df[df['restaurant_available'] == True].copy()

print(f'\n✅ Total: {len(df)} registros | {len(df_valid)} válidos ({100*len(df_valid)//len(df)}% disponibilidad)')

In [ ]:
# Así luce el retry logic en el BaseScraper (extracto del código real)
retry_code = '''
async def _navigate(self, url: str, wait_until: str = "networkidle") -> bool:
    """Navega a una URL con retry logic y rate limiting."""
    for attempt in range(1, self.MAX_RETRIES + 1):       # MAX_RETRIES = 3
        try:
            await self._rate_limit()                      # mínimo 2s + jitter
            await self.page.goto(url, wait_until=wait_until, timeout=45_000)
            return True
        except Exception as e:
            logger.warning(f"Intento {attempt}/{self.MAX_RETRIES} falló: {e}")
            if attempt < self.MAX_RETRIES:
                backoff = 2 ** attempt + random.uniform(0, 1)  # 2s, 4s, 8s
                await asyncio.sleep(backoff)
    logger.error(f"Falló definitivamente después de {self.MAX_RETRIES} intentos")
    return False
'''
print(retry_code)

# Simular el backoff exponencial
import random
print('Ejemplo de backoff exponencial (3 intentos):')
for attempt in range(1, 4):
    backoff = 2**attempt + random.uniform(0, 1)
    print(f'  Intento {attempt} falló → esperar {backoff:.1f}s antes del siguiente')

<a id='4'></a>

---
# 4. Datos Recolectados — Overview

### ¿Qué capturamos por cada observación?

| Campo | Descripción |
|---|---|
| `price_product` | Precio del ítem en la plataforma (incluye markup de la app) |
| `delivery_fee` | Costo de envío antes de descuentos |
| `service_fee` | Fee de servicio (% sobre subtotal, estimado desde checkout público) |
| `estimated_delivery_min/max` | Rango de tiempo de entrega en minutos |
| `discounts_active` | Lista de promociones activas visibles en la sesión |
| `restaurant_available` | Si el restaurante está disponible en esa zona |
| `final_price_total` | Costo total que paga el usuario (producto + delivery + service fee) |

In [ ]:
# Resumen estadístico por plataforma
df_valid['platform_label'] = df_valid['platform'].map(LABELS)
df_valid['eta_mid'] = (df_valid['estimated_delivery_min'] + df_valid['estimated_delivery_max']) / 2

summary = df_valid.groupby('platform').agg(
    registros=('price_product', 'count'),
    precio_bigmac=('price_product', lambda x: x[df_valid.loc[x.index, 'product_id'] == 'bigmac'].mean()),
    delivery_fee_prom=('delivery_fee', 'mean'),
    service_fee_prom=('service_fee', 'mean'),
    eta_prom_min=('eta_mid', 'mean'),
    pct_envio_gratis=('delivery_fee', lambda x: (x == 0).mean() * 100),
).round(2)

summary.index = [LABELS[p] for p in summary.index]
summary.columns = ['Registros', 'Big Mac ($)', 'Delivery fee ($)', 'Service fee ($)', 'ETA (min)', '% Envío gratis']
print('Resumen estadístico por plataforma\n')
print(summary.to_string())

In [ ]:
# Distribución de precios del Big Mac — las 3 plataformas
df_bm = df_valid[df_valid['product_id'] == 'bigmac'].copy()

fig = go.Figure()
for platform, label in LABELS.items():
    data = df_bm[df_bm['platform'] == platform]['price_product']
    fig.add_trace(go.Box(
        y=data, name=label,
        marker_color=COLORS[platform],
        boxmean='sd',
        hovertemplate='%{y:.0f} MXN'
    ))

fig.update_layout(
    title='Distribución de precios — Big Mac por plataforma (todas las zonas)',
    yaxis_title='Precio (MXN)',
    height=420,
    showlegend=True
)
fig.show()

# Disponibilidad por plataforma y ciudad
avail = df.groupby(['platform', 'city']).apply(
    lambda x: x['restaurant_available'].mean() * 100
).reset_index(name='disponibilidad')
avail['platform_label'] = avail['platform'].map(LABELS)

fig2 = px.bar(avail, x='city', y='disponibilidad', color='platform_label',
    barmode='group',
    color_discrete_map={v: COLORS[k] for k, v in LABELS.items()},
    labels={'city': 'Ciudad', 'disponibilidad': '% disponibilidad', 'platform_label': 'Plataforma'},
    title='Disponibilidad de McDonald\'s por ciudad y plataforma (%)',
    text=avail['disponibilidad'].apply(lambda v: f'{v:.0f}%'))
fig2.update_traces(textposition='outside')
fig2.update_layout(height=380, yaxis_range=[0, 110])
fig2.show()

<a id='5'></a>

---
# 5. Top 5 Insights Accionables

> Cada insight incluye el **finding** (qué encontramos), el **impacto** (por qué importa para el negocio) y la **recomendación** concreta para Rappi.

In [ ]:
# ─── INSIGHT 1: Desventaja en delivery fees en zonas de expansión ─────────────
print('=' * 65)
print('  INSIGHT #1 — Desventaja en delivery fees en zonas de expansión')
print('=' * 65)

popular_fees = df_valid[df_valid['zone_type'] == 'popular'].groupby('platform')['delivery_fee'].mean().round(2)
rappi_fee = popular_fees.get('rappi', 0)
didi_fee  = popular_fees.get('didifood', 0)
diff_pct  = (rappi_fee - didi_fee) / rappi_fee * 100

print(f'\n📊 FINDING:')
print(f'   DiDi Food cobra ${didi_fee:.0f} de delivery fee en zonas populares')
print(f'   Rappi cobra ${rappi_fee:.0f} — {diff_pct:.0f}% más caro')

print(f'\n⚡ IMPACTO:')
print(f'   Zonas populares (Iztapalapa, Ecatepec, Apodaca) = mayor potencial')
print(f'   de crecimiento. Usuarios sensibles al precio deciden por el fee.')

print(f'\n✅ RECOMENDACIÓN:')
print(f'   Subsidio dinámico de fee en horario pico (12-14h y 19-21h) en zonas')
print(f'   clave. Meta: paridad con DiDi Food en zonas popular para Q2.')

# Gráfico
zone_order = ['high_income', 'mid_high', 'mid', 'popular']
zone_labels_map = {'high_income': 'Alto ingreso', 'mid_high': 'Medio-alto', 'mid': 'Medio', 'popular': 'Popular'}

fees_by_zone = df_valid.groupby(['platform', 'zone_type'])['delivery_fee'].mean().round(2).reset_index()
fees_by_zone['zone_label'] = fees_by_zone['zone_type'].map(zone_labels_map)
fees_by_zone['platform_label'] = fees_by_zone['platform'].map(LABELS)
fees_by_zone['fee_label'] = fees_by_zone['delivery_fee'].apply(lambda v: f'${v:.0f}')
fees_by_zone = fees_by_zone[fees_by_zone['zone_type'].isin(zone_order)]

fig = px.bar(fees_by_zone, x='zone_label', y='delivery_fee', color='platform_label',
    barmode='group',
    color_discrete_map={v: COLORS[k] for k, v in LABELS.items()},
    category_orders={'zone_label': [zone_labels_map[z] for z in zone_order]},
    labels={'zone_label': 'Tipo de zona', 'delivery_fee': 'Delivery fee (MXN)', 'platform_label': 'Plataforma'},
    title='#1 · Delivery fee promedio por zona — Rappi vs competencia',
    text='fee_label')
fig.update_traces(textposition='outside')
fig.update_layout(height=420)
fig.show()

In [ ]:
# ─── INSIGHT 2: Uber Eats más rápido en zonas premium de CDMX ────────────────
print('=' * 65)
print('  INSIGHT #2 — Uber Eats más rápido en zonas premium de CDMX')
print('=' * 65)

df_premium = df_valid[(df_valid['city'] == 'CDMX') & (df_valid['zone_type'] == 'high_income')].copy()
etas = df_premium.groupby('platform')['eta_mid'].mean().round(1)
rappi_eta = etas.get('rappi', 0)
uber_eta  = etas.get('ubereats', 0)
gap = rappi_eta - uber_eta

print(f'\n📊 FINDING:')
print(f'   Uber Eats: {uber_eta:.0f} min promedio en Polanco/Santa Fe/Condesa')
print(f'   Rappi: {rappi_eta:.0f} min — {gap:.0f} minutos más lento')

print(f'\n⚡ IMPACTO:')
print(f'   Usuarios premium son más sensibles al tiempo que al precio.')
print(f'   Son los usuarios de mayor ticket promedio — riesgo de churn.')

print(f'\n✅ RECOMENDACIÓN:')
print(f'   Optimizar algoritmo de asignación de repartidores en zonas premium.')
print(f'   Incentivo de tarifa preferente para repartidores en Polanco/Santa Fe.')
print(f'   Target: <20 min en zonas high_income para Q3.')

# Gráfico
eta_data = df_valid.groupby(['platform', 'city'])['eta_mid'].mean().round(1).reset_index()
eta_data['platform_label'] = eta_data['platform'].map(LABELS)
eta_data['eta_label'] = eta_data['eta_mid'].apply(lambda v: f'{v:.0f} min')
city_order = ['CDMX', 'Ecatepec', 'Tlalnepantla', 'Guadalajara', 'Monterrey']

fig = px.bar(eta_data, x='city', y='eta_mid', color='platform_label',
    barmode='group',
    color_discrete_map={v: COLORS[k] for k, v in LABELS.items()},
    category_orders={'city': city_order},
    labels={'city': 'Ciudad', 'eta_mid': 'Tiempo estimado (min)', 'platform_label': 'Plataforma'},
    title='#2 · Tiempo de entrega promedio por ciudad — Rappi vs competencia',
    text='eta_label')
fig.update_traces(textposition='outside')
fig.update_layout(height=420)
fig.show()

In [ ]:
# ─── INSIGHT 3: Estructura de fees — Rappi el más caro en total ──────────────
print('=' * 65)
print('  INSIGHT #3 — Rappi tiene el mayor costo total al usuario')
print('=' * 65)

df_bm = df_valid[df_valid['product_id'] == 'bigmac'].copy()
fee_struct = df_bm.groupby('platform')[['price_product', 'delivery_fee', 'service_fee', 'final_price_total']].mean().round(2)

for platform, row in fee_struct.iterrows():
    label = LABELS[platform]
    print(f'\n  {label}:')
    print(f'    Precio producto : ${row["price_product"]:.2f}')
    print(f'    Delivery fee    : ${row["delivery_fee"]:.2f}')
    print(f'    Service fee     : ${row["service_fee"]:.2f}')
    print(f'    TOTAL           : ${row["final_price_total"]:.2f}')

rappi_total = fee_struct.loc['rappi', 'final_price_total']
didi_total  = fee_struct.loc['didifood', 'final_price_total']
print(f'\n📊 FINDING: Rappi es ${rappi_total - didi_total:.0f} MXN más caro que DiDi Food por el mismo Big Mac')
print(f'\n✅ RECOMENDACIÓN: Negociar price match con McDonald\'s en los 3 ítems más pedidos')
print(f'   o absorber diferencial con Rappi Credits en primeros 3 pedidos del mes.')

# Gráfico stacked
fig = go.Figure()
components = [('price_product', 'Precio producto'), ('delivery_fee', 'Delivery fee'), ('service_fee', 'Service fee')]
for col, label in components:
    fig.add_trace(go.Bar(
        name=label,
        x=[LABELS[p] for p in fee_struct.index],
        y=fee_struct[col].values,
        text=[f'${v:.0f}' for v in fee_struct[col].values],
        textposition='inside'
    ))

for i, (p, row) in enumerate(fee_struct.iterrows()):
    fig.add_annotation(x=LABELS[p], y=row['final_price_total'],
        text=f"<b>${row['final_price_total']:.0f}</b>",
        showarrow=False, yshift=12, font=dict(size=13))

fig.update_layout(barmode='stack',
    title='#3 · Estructura de costo total al usuario — Big Mac',
    yaxis_title='Costo (MXN)', height=430)
fig.show()

In [ ]:
# ─── INSIGHT 4: DiDi domina Guadalajara con envío gratis ─────────────────────
print('=' * 65)
print('  INSIGHT #4 — DiDi Food captura Guadalajara con envío gratis')
print('=' * 65)

gdl_free = df_valid[df_valid['city'] == 'Guadalajara'].groupby('platform').apply(
    lambda x: (x['delivery_fee'] == 0).mean() * 100
).round(1)

for p, pct in gdl_free.items():
    print(f'  {LABELS[p]:<15}: {pct:.0f}% de casos con envío gratis en GDL')

print(f'\n📊 FINDING: DiDi ofrece envío gratis en {gdl_free.get("didifood",0):.0f}% de los casos en GDL')
print(f'            vs {gdl_free.get("rappi",0):.0f}% de Rappi — estrategia clara de market capture')
print(f'\n⚡ IMPACTO: GDL es el 2do mercado más grande de Rappi en México.')
print(f'            El efecto de red hace muy difícil recuperar usuarios migrados.')
print(f'\n✅ RECOMENDACIÓN: Campaña "Rappi Gratis en GDL" — envío gratis garantizado')
print(f'   las primeras 4 semanas del mes. Evaluar sostenibilidad a 60 días.')

# Gráfico
free_data = df_valid.groupby(['platform', 'city']).apply(
    lambda x: (x['delivery_fee'] == 0).mean() * 100
).round(1).reset_index(name='free_pct')
free_data['platform_label'] = free_data['platform'].map(LABELS)
free_data['pct_label'] = free_data['free_pct'].apply(lambda v: f'{v:.0f}%')

fig = px.bar(free_data, x='city', y='free_pct', color='platform_label',
    barmode='group',
    color_discrete_map={v: COLORS[k] for k, v in LABELS.items()},
    category_orders={'city': city_order},
    labels={'city': 'Ciudad', 'free_pct': '% con envío gratis', 'platform_label': 'Plataforma'},
    title='#4 · Porcentaje de casos con envío gratis por ciudad',
    text='pct_label')
fig.update_traces(textposition='outside')
fig.update_layout(height=420, yaxis_range=[0, 110])
fig.show()

In [ ]:
# ─── INSIGHT 5: Rappi tiene más promos pero menor visibilidad ────────────────
print('=' * 65)
print('  INSIGHT #5 — Rappi tiene más promos pero menor conversión visible')
print('=' * 65)

df_valid['n_discounts'] = df_valid['discounts_active'].apply(len)
promo_avg = df_valid.groupby('platform')['n_discounts'].mean().round(1)

for p, avg in promo_avg.items():
    print(f'  {LABELS[p]:<15}: {avg:.1f} promociones activas promedio por sesión')

print(f'\n📊 FINDING: Rappi tiene más promociones ({promo_avg.get("rappi",0):.1f}/sesión) que')
print(f'            Uber Eats ({promo_avg.get("ubereats",0):.1f}) pero la UX las oculta hasta el checkout.')
print(f'\n⚡ IMPACTO: Invertir en promos que el usuario no ve antes de decidir = dinero quemado.')
print(f'\n✅ RECOMENDACIÓN: A/B test — banner "Ahorras $X hoy" en pantalla de inicio.')
print(f'   Target: +15% en tasa de uso de cupones.')

# Clasificar tipos de promo
def classify_promo(text):
    t = text.lower()
    if 'envío gratis' in t or 'envio gratis' in t: return 'Envío gratis'
    if '%' in t and ('descuento' in t or 'off' in t): return '% de descuento'
    if 'prime' in t or 'one' in t or 'suscripci' in t: return 'Suscripción/membresía'
    if 'primer pedido' in t or 'primera compra' in t: return 'Descuento primer pedido'
    if 'cupón' in t or 'cupon' in t or 'código' in t: return 'Cupón / código'
    if '$' in t and 'descuento' in t: return 'Descuento fijo ($)'
    return 'Otra promoción'

df_exploded = df_valid.explode('discounts_active').dropna(subset=['discounts_active'])
df_exploded = df_exploded[df_exploded['discounts_active'].str.strip() != '']
df_exploded['promo_cat'] = df_exploded['discounts_active'].apply(classify_promo)

total_sessions = df_valid.groupby('platform').size()
counts = df_exploded.groupby(['platform', 'promo_cat']).size().reset_index(name='count')
counts['pct'] = counts.apply(lambda r: round(r['count'] / total_sessions.get(r['platform'], 1) * 100, 1), axis=1)
counts['platform_label'] = counts['platform'].map(LABELS)
counts['pct_label'] = counts['pct'].apply(lambda v: f'{v:.1f}%')
cat_order = counts.groupby('promo_cat')['pct'].sum().sort_values(ascending=True).index.tolist()

fig = px.bar(counts, x='pct', y='promo_cat', color='platform_label',
    orientation='h', barmode='group',
    color_discrete_map={v: COLORS[k] for k, v in LABELS.items()},
    category_orders={'promo_cat': cat_order},
    labels={'pct': '% de sesiones', 'promo_cat': 'Tipo de promoción', 'platform_label': 'Plataforma'},
    title='#5 · Estrategia promocional — tipos de descuento por plataforma',
    text='pct_label')
fig.update_traces(textposition='outside')
fig.update_layout(height=420, xaxis=dict(range=[0, counts['pct'].max() * 1.25]))
fig.show()

<a id='6'></a>

---
# 6. Decisiones Técnicas y Desafíos

## ¿Por qué Playwright y no Scrapy o requests?

| Herramienta | Por qué no |
|---|---|
| `requests` + BeautifulSoup | Rappi, Uber Eats y DiDi son **SPAs React** — el HTML inicial no tiene datos, todo se carga vía JS |
| Scrapy | No ejecuta JavaScript — mismo problema |
| Selenium | Más lento y más fácil de detectar que Playwright |
| **Playwright** ✅ | Async nativo, CDP directo, mejor stealth, soporte para múltiples contextos |

## Desafíos enfrentados y soluciones

In [ ]:
challenges = [
    {
        'desafío': 'Detección de headless browser',
        'plataforma': 'Uber Eats (Cloudflare), Rappi',
        'solución': 'Inyección de init_script para ocultar navigator.webdriver, plugins, chrome.runtime. User-agent de Chrome real. Locale es-MX + geolocalización CDMX.',
        'código': "await self.context.add_init_script(\"\"\"\n  Object.defineProperty(navigator, 'webdriver', { get: () => undefined });\n  Object.defineProperty(navigator, 'plugins', { get: () => [1,2,3,4,5] });\n  window.chrome = { runtime: {} };\n\"\"\")"
    },
    {
        'desafío': 'Selectores CSS inestables (SPA React)',
        'plataforma': 'Todas',
        'solución': 'Múltiples selectores en lista de fallback. Si falla el primero, intenta el siguiente.',
        'código': "SEL_DELIVERY_FEE = [\n  '[data-testid=\"store-meta-delivery-fee\"]',\n  '[class*=\"DeliveryFee\"]',\n  'span[aria-label*=\"envío\"]',  # fallback\n]"
    },
    {
        'desafío': 'Service fee no visible sin checkout',
        'plataforma': 'Todas',
        'solución': 'Estimado como % conocido del subtotal (Rappi ~8%, Uber ~6%, DiDi ~4%) basado en valores públicos reportados por usuarios.',
        'código': "service_fee = round(price_product * cfg['service_fee_pct'], 2)  # estimado"
    },
    {
        'desafío': 'Geolocalización incorrecta sin IP mexicana',
        'plataforma': 'Uber Eats, DiDi Food',
        'solución': 'Playwright context con timezone America/Mexico_City + geolocation CDMX. Para cobertura completa: soporte para PROXY_URL en .env.',
        'código': "context = await browser.new_context(\n  locale='es-MX',\n  timezone_id='America/Mexico_City',\n  geolocation={'latitude': 19.4326, 'longitude': -99.1332},\n)"
    },
]

for i, c in enumerate(challenges, 1):
    print(f'\n{'─'*65}')
    print(f'  Desafío #{i}: {c["desafío"]}')
    print(f'  Plataforma: {c["plataforma"]}')
    print(f'\n  Solución: {c["solución"]}')
    print(f'\n  Código:')
    for line in c['código'].split('\n'):
        print(f'    {line}')

In [ ]:
# Rate limiting — demostración del mecanismo
import time, random, asyncio

print('Rate limiting implementado:\n')
print('  - Mínimo 2.0s entre requests al mismo dominio')
print('  - Jitter aleatorio (+0.3s a +1.2s) para parecer humano')
print('  - Lock compartido entre instancias del mismo dominio')
print('  - Pausa adicional de 1.5-4.0s entre direcciones distintas')

# Simulación
print('\nSimulación de tiempos entre requests (10 requests a ubereats.com):')
MIN_DELAY = 2.0
last_time = 0
for i in range(1, 11):
    jitter = random.uniform(0.3, 1.2)
    wait = MIN_DELAY + jitter
    print(f'  Request {i:2d}: espera {wait:.2f}s antes de enviar')

print(f'\n  → Tiempo total mínimo para 10 requests: ~{10 * (MIN_DELAY + 0.75):.0f}s')
print(f'  → Esto protege los servidores y reduce probabilidad de bloqueo')

In [ ]:
# Test suite — ejecutar los tests y mostrar resultados
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True, encoding='utf-8'
)
print(result.stdout)
if result.returncode == 0:
    print('✅ Todos los tests pasaron')
else:
    print('❌ Algunos tests fallaron')
    print(result.stderr)

<a id='7'></a>

---
# 7. Limitaciones y Next Steps

## Limitaciones conocidas

### Técnicas

| Limitación | Impacto | Mitigación actual |
|---|---|---|
| Requiere IP mexicana para cobertura real | Sin proxy, algunos restaurantes no aparecen disponibles | Soporte para `PROXY_URL` en `.env` |
| Selectores CSS cambian frecuentemente (SPA) | El scraper puede dejar de funcionar sin previo aviso | Lista de selectores fallback por cada elemento |
| Service fee solo visible en checkout | No se puede scrapear sin simular una compra completa | Estimado con % basado en valores públicos conocidos |
| DiDi Food tiene menor cobertura en zonas periféricas | Algunos registros marcan unavailable en Ecatepec/Escobedo | Documentado en limitaciones, tracked en `restaurant_available` |

### De datos
- **Snapshot en el tiempo**: los precios cambian dinámicamente — el sistema captura un momento, no tendencias
- **Variabilidad por horario**: las plataformas aplican surge pricing — sería valioso comparar precio en distintos horarios
- **Muestra de productos**: 5 ítems de referencia cubren fast food y retail, pero no farma, supermercado o dark kitchens

## Next Steps — ¿Qué haría con más tiempo?

### Corto plazo (1-2 semanas)
1. **Proxy pool integrado** — ScraperAPI o BrightData para scraping real desde IPs mexicanas sin depender de VPN manual
2. **Scheduler con cron** — ejecutar el scraper 3 veces al día (mañana / mediodía / noche) para capturar surge pricing
3. **Alertas automáticas** — notificación por Slack/email cuando DiDi baja su fee en una zona clave

### Mediano plazo (1 mes)
4. **Ampliar productos** — incluir categorías de dark kitchens propias de cada plataforma (Rappi Now, DiDi Mart)
5. **Análisis de tendencias** — dashboard de series de tiempo para ver cómo evolucionan precios y fees semana a semana
6. **Cobertura de restaurantes** — extender a cadenas más allá de McDonald's (Burger King, Domino's) para validar si el markup es sistemático

### Largo plazo
7. **Modelo predictivo** — predecir dónde DiDi aumentará su agresividad de pricing basado en patrones históricos
8. **Integración con datos internos de Rappi** — cruzar con datos de conversión para cuantificar el impacto de cada diferencial de precio

In [ ]:
# Resumen ejecutivo final
print('=' * 65)
print('  RESUMEN EJECUTIVO')
print('=' * 65)

df_valid['n_discounts'] = df_valid['discounts_active'].apply(len)
bm = df_valid[df_valid['product_id'] == 'bigmac'].groupby('platform')

for platform in ['rappi', 'ubereats', 'didifood']:
    grp = bm.get_group(platform)
    print(f'\n  {LABELS[platform]}')
    print(f'    Big Mac           : ${grp["price_product"].mean():.2f} MXN')
    print(f'    Delivery fee prom : ${df_valid[df_valid["platform"]==platform]["delivery_fee"].mean():.2f} MXN')
    print(f'    Service fee prom  : ${df_valid[df_valid["platform"]==platform]["service_fee"].mean():.2f} MXN')
    print(f'    ETA promedio      : {df_valid[df_valid["platform"]==platform]["eta_mid"].mean():.0f} min')
    print(f'    % Envío gratis    : {(df_valid[df_valid["platform"]==platform]["delivery_fee"]==0).mean()*100:.0f}%')
    print(f'    Promos/sesión     : {df_valid[df_valid["platform"]==platform]["n_discounts"].mean():.1f}')

print('\n' + '=' * 65)
print('  Top 3 acciones recomendadas para Rappi:')
print('    1. Subsidio de delivery fee en zonas populares (Ecatepec, Iztapalapa)')
print('    2. Campaña Rappi Gratis en GDL para frenar a DiDi')
print('    3. Rediseñar visibilidad de promos antes del checkout')
print('=' * 65)